In [7]:
!pip install -U openpyxl XlsxWriter

In [8]:
# -*- coding: utf-8 -*-
import os, re, unicodedata
from glob import glob
import numpy as np
import pandas as pd

# ===== 설정 =====
DATA_DIR = "data"          # 입력 엑셀들이 있는 폴더
OUT_XLSX = "읍면동_시간대별_이용량_통합_2023_2024.xlsx"
YEARS    = {2023, 2024}
MERGE_WITH_EXISTING = True      # 기존 결과 파일과 병합(있을 때)

METRO_MAP = ["서울","부산","대구","인천","광주","대전","울산","세종","경기",
             "강원","충북","충남","전북","전남","경북","경남","제주"]
NFC_METRO = {unicodedata.normalize("NFC", m): m for m in METRO_MAP}

# 참조 CSV와 동일한 시간대 순서: 04~23 → 00~03
HOURS_ORDER = [f"{h:02d}" for h in list(range(4,24)) + list(range(0,4))]
WIDE_COLS = [f"{p}_{hh}시" for hh in HOURS_ORDER for p in ("발생량","도착량")]
KEYS = ["시도","시군구","읍면동","년도"]

# ===== 대상 파일 검색(지역+연도) =====
def list_year_region_excels(data_dir, years=YEARS):
    """
    파일명: ^(지역명)([_\\-\\s]?)(연도)\\.xlsx$
    예) 서울_2023.xlsx, 울산2024.xlsx, 인천-2024.xlsx, 경기 2023.xlsx
    """
    region_alt = "|".join(map(re.escape, METRO_MAP))
    year_alt = "|".join(map(str, sorted(years)))
    pat = re.compile(fr"^({region_alt})(?:[_\-\s]?)(?:({year_alt}))\.xlsx$", re.IGNORECASE)

    hits = []
    for p in glob(os.path.join(data_dir, "*.xlsx")):
        name = os.path.basename(p)
        m = pat.match(name)
        if not m:
            continue
        region_raw, year_str = m.group(1), m.group(2)
        region = NFC_METRO.get(unicodedata.normalize("NFC", region_raw), region_raw)
        hits.append((p, region, int(year_str)))

    order = {r: i for i, r in enumerate(METRO_MAP)}
    hits.sort(key=lambda x: (order.get(x[1], 999), x[2], x[0].lower()))
    return hits

# ===== 파서 =====
def parse_one(path: str) -> pd.DataFrame:
    try:
        df0 = pd.read_excel(path, sheet_name="Main View")
    except Exception:
        xl = pd.ExcelFile(path)
        df0 = pd.read_excel(path, sheet_name=xl.sheet_names[0])

    # 0행 서브헤더(시도/시군구/읍면동/년/발생량/도착량)를 이용해 새 컬럼명 구성
    new_cols = []
    for i, col in enumerate(df0.columns):
        sub = str(df0.iloc[0, i]).strip() if pd.notna(df0.iloc[0, i]) else ""

        if i == 0 and sub == "시도코드":
            new_cols.append("시도코드")
        elif sub in ("시도","시군구","읍면동","년"):
            new_cols.append("년도" if sub == "년" else sub)
        elif col == "합계":
            if sub == "발생량":
                new_cols.append("합계_발생량")
            elif sub == "도착량":
                new_cols.append("합계_도착량")
            else:
                new_cols.append(f"합계_{sub or i}")
        else:
            # 시간대 컬럼: col이 '04', 10, '00' 등 → 시 문자열
            try:
                hour_str = f"{int(col):02d}"
            except Exception:
                s = str(col).strip()
                hour_str = f"{int(s):02d}" if s.isdigit() else s
            if sub in ("발생량","도착량"):
                new_cols.append(f"{sub}_{hour_str}시")
            else:
                new_cols.append(f"{hour_str}_{sub or i}")

    df0.columns = new_cols
    df = df0.iloc[1:].reset_index(drop=True)

    base_cols = ["시도","시군구","읍면동","년도"]
    numeric_cols = [c for c in df.columns if c.startswith("발생량_") or c.startswith("도착량_")]
    keep = [c for c in base_cols if c in df.columns] + numeric_cols
    df = df[keep].copy()

    for c in ("시도","시군구"):
        if c in df.columns:
            df[c] = df[c].replace({0: np.nan, "0": np.nan}).ffill()

    if "년도" in df.columns:
        df["년도"] = pd.to_numeric(df["년도"], errors="coerce")
        df = df[df["년도"].isin(YEARS)]

    if "읍면동" in df.columns:
        df["읍면동"] = df["읍면동"].astype(str).str.strip()
        df = df[df["읍면동"].notna() & (df["읍면동"] != "") & (df["읍면동"] != "0")]

    for c in WIDE_COLS:
        if c not in df.columns:
            df[c] = 0

    ordered = KEYS + WIDE_COLS
    df = df[ordered]

    for c in WIDE_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype("int64")

    df["__source_file__"] = os.path.basename(path)
    return df

# ===== 실행 (재스캔 + (옵션)기존 결과 병합) =====
targets = list_year_region_excels(DATA_DIR, years=YEARS)
if not targets:
    raise FileNotFoundError("대상 파일이 없습니다. 예) 서울_2023.xlsx, 울산2024.xlsx 등")

frames = [parse_one(p) for p, _, _ in targets]
new_df = pd.concat(frames, ignore_index=True)

final_df = new_df.copy()

if MERGE_WITH_EXISTING and os.path.exists(OUT_XLSX):
    try:
        old_df = pd.read_excel(OUT_XLSX, sheet_name="통합")
        # 키가 겹치는 행은 새 데이터로 교체, 나머지 키는 유지
        key_df = new_df[KEYS].drop_duplicates()
        tmp = old_df.merge(key_df, on=KEYS, how="left", indicator=True)
        old_only = tmp[tmp["_merge"] == "left_only"].drop(columns=["_merge"])
        # 컬럼 순서 맞추기
        old_only = old_only.reindex(columns=final_df.columns, fill_value=0)
        final_df = pd.concat([old_only, new_df], ignore_index=True)
    except Exception:
        # 문제가 있으면 새 데이터만 사용
        pass

# ===== 저장 =====
with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as writer:
    final_df.drop(columns=["__source_file__"], errors="ignore").to_excel(writer, sheet_name="통합", index=False)

    if {"시도","년도"}.issubset(final_df.columns):
        for (sido, year), grp in final_df.groupby(["시도","년도"], as_index=False):
            name = f"{str(sido)[:6]}_{int(year)}"[:31]
            grp.drop(columns=["__source_file__"], errors="ignore").to_excel(writer, sheet_name=name, index=False)

print(f"[DONE] 생성/갱신: {OUT_XLSX}")

[DONE] 생성/갱신: 읍면동_시간대별_이용량_통합_2023_2024.xlsx


In [9]:
# -*- coding: utf-8 -*-
# IN_XLSX(통합 와이드) → 롱 2종(원본전개/중복합산) 생성
# OUT_XLSX가 이미 있으면 키 기준으로 기존 행을 교체(replace)하고, 없으면 새로 생성
import pandas as pd
import numpy as np
import os

IN_XLSX  = "읍면동_시간대별_이용량_통합_2023_2024.xlsx"
OUT_XLSX = "읍면동 최종 이용량_통합.xlsx"

KEYS = ["시도","시군구","읍면동","년도","시간대"]
hour_labels = [f"{h:02d}시" for h in range(24)]
occ_cols = [f"발생량_{h}" for h in hour_labels]
arr_cols = [f"도착량_{h}" for h in hour_labels]

def to_long_tables(usage_wide: pd.DataFrame):
    # 시간대 컬럼 보정
    for c in occ_cols + arr_cols:
        if c not in usage_wide.columns:
            usage_wide[c] = 0

    # (A) 원본전개: 와이드 → 롱
    blocks = []
    base = usage_wide[["시도","시군구","읍면동","년도"]].copy()
    for h in hour_labels:
        b = base.copy()
        b["시간대"] = h
        b["발생량"] = pd.to_numeric(usage_wide[f"발생량_{h}"], errors="coerce").fillna(0).astype("int64")
        b["도착량"] = pd.to_numeric(usage_wide[f"도착량_{h}"], errors="coerce").fillna(0).astype("int64")
        b["이용량"] = (b["발생량"] + b["도착량"]).astype("int64")
        blocks.append(b)
    long_usage = pd.concat(blocks, ignore_index=True)

    # (B) 중복합산: 같은 (시도,시군구,읍면동,년도) 중복을 시간대별 합산 후 롱
    num_cols = occ_cols + arr_cols
    agg = usage_wide.groupby(["시도","시군구","읍면동","년도"], as_index=False)[num_cols].sum()

    blocks2 = []
    base2 = agg[["시도","시군구","읍면동","년도"]].copy()
    for h in hour_labels:
        b = base2.copy()
        b["시간대"] = h
        b["발생량"] = agg[f"발생량_{h}"].astype("int64")
        b["도착량"] = agg[f"도착량_{h}"].astype("int64")
        b["이용량"] = (b["발생량"] + b["도착량"]).astype("int64")
        blocks2.append(b)
    long_usage_dedup = pd.concat(blocks2, ignore_index=True)

    return long_usage, long_usage_dedup

def replace_by_keys(old_df: pd.DataFrame, new_df: pd.DataFrame, keys=KEYS) -> pd.DataFrame:
    """기존 DF에서 new_df와 키가 겹치는 행은 제거하고, new_df로 교체 후 결합"""
    if old_df is None or old_df.empty:
        return new_df.copy()
    key_df = new_df[keys].drop_duplicates()
    merged = old_df.merge(key_df, on=keys, how="left", indicator=True)
    old_only = merged.loc[merged["_merge"]=="left_only"].drop(columns=["_merge"])
    # 열 순서/결측 보정
    old_only = old_only.reindex(columns=new_df.columns, fill_value=0)
    out = pd.concat([old_only, new_df], ignore_index=True)
    return out

# 1) 원본 통합(와이드) 로드
usage = pd.read_excel(IN_XLSX, sheet_name="통합", engine="openpyxl")

# 2) 롱 테이블 2종 생성
long_usage, long_usage_dedup = to_long_tables(usage)

# 3) 기존 OUT_XLSX가 있으면 읽어서 교체 병합
exist_original, exist_dedup = None, None
if os.path.exists(OUT_XLSX):
    with pd.ExcelFile(OUT_XLSX, engine="openpyxl") as xf:
        if "원본전개" in xf.sheet_names:
            exist_original = pd.read_excel(xf, sheet_name="원본전개")
        if "중복합산" in xf.sheet_names:
            exist_dedup = pd.read_excel(xf, sheet_name="중복합산")

final_original = replace_by_keys(exist_original, long_usage, KEYS)
final_dedup    = replace_by_keys(exist_dedup, long_usage_dedup, KEYS)

# 4) 엑셀로만 저장 (시트 2개)
with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as writer:
    final_original.to_excel(writer, sheet_name="원본전개", index=False)
    final_dedup.to_excel(writer, sheet_name="중복합산", index=False)

print(f"[DONE] 갱신 저장: {OUT_XLSX}")


[DONE] 갱신 저장: 읍면동 최종 이용량_통합.xlsx


In [10]:
# -*- coding: utf-8 -*-
# 입력:  읍면동 최종 이용량_통합.xlsx (시트: 중복합산/원본전개)
# 출력:  혼잡도 최종등급_통합.xlsx (엑셀만 저장)

import os
import numpy as np
import pandas as pd

IN_XLSX = "읍면동 최종 이용량_통합.xlsx"
OUT_XLSX = "혼잡도 최종등급_통합.xlsx"

PREF_SHEET, FALLBACK_SHEET = "중복합산", "원본전개"

# 혼잡도 등급 매핑
LEVEL_LABEL = {1: "여유", 2: "보통", 3: "붐빔", 4: "매우 붐빔"}

# 혼잡도 계산 기준
# - 상대평가 그룹(권장): 같은 시군구·같은 연도·같은 시간대 내에서 상대 비교
GROUP_KEYS = ["시도", "시군구", "년도", "시간대"]
# - fixed 기준(절대 baseline): 시도·연도별 전역 분포 기준
FIXED_BASELINE_KEYS = ["시도", "년도"]

# 최종 점수 결합 가중치 (quantile 0.6 + fixed 0.4)
W_Q, W_F = 0.6, 0.4

# 1) 로드 (중복합산 우선)
try:
    df = pd.read_excel(IN_XLSX, sheet_name=PREF_SHEET)
except Exception:
    df = pd.read_excel(IN_XLSX, sheet_name=FALLBACK_SHEET)

# 2) 컬럼/수치 보정
need = {"시도", "시군구", "읍면동", "년도", "시간대"}
miss = need - set(df.columns)
if miss:
    raise ValueError(f"입력에 필수 컬럼 누락: {sorted(miss)}")

# 이용량 생성/보정
if "이용량" not in df.columns:
    occ = pd.to_numeric(df.get("발생량", 0), errors="coerce").fillna(0)
    arr = pd.to_numeric(df.get("도착량", 0), errors="coerce").fillna(0)
    df["이용량"] = (occ + arr).astype(float)
else:
    df["이용량"] = pd.to_numeric(df["이용량"], errors="coerce").fillna(0.0)

# 3) 그룹별(z, min-max, quantile) 계산
def _group_metrics(g: pd.DataFrame) -> pd.DataFrame:
    x = g["이용량"].astype(float).values
    # z-score
    m, s = np.mean(x), np.std(x, ddof=0)
    z = (x - m) / s if s > 0 else np.zeros_like(x)
    # min-max
    mn, mx = np.min(x), np.max(x)
    mm = (x - mn) / (mx - mn) if mx > mn else np.zeros_like(x)
    # quantile level(1~4)
    if len(x) > 1:
        q1, q2, q3 = np.quantile(x, [0.25, 0.5, 0.75])
    else:
        q1 = q2 = q3 = 0
    q_level = np.select(
        [x <= q1, (x > q1) & (x <= q2), (x > q2) & (x <= q3), x > q3],
        [1, 2, 3, 4],
        default=1
    ).astype(int)

    out = g.copy()
    out["z_score"] = z
    out["min_max"] = mm
    out["quantile_level_num"] = q_level
    out["quantile_level"] = out["quantile_level_num"].map(LEVEL_LABEL)
    return out

df = df.groupby(GROUP_KEYS, group_keys=False).apply(_group_metrics)

# 4) fixed 기준(시도·연도별 전역 분포로 절대 기준선) → level(1~4)
def _fixed_baseline(g: pd.DataFrame) -> pd.DataFrame:
    x = g["이용량"].astype(float).values
    if len(x) > 1:
        q1, q2, q3 = np.quantile(x, [0.25, 0.5, 0.75])
    else:
        q1 = q2 = q3 = 0
    g["_fq1"], g["_fq2"], g["_fq3"] = q1, q2, q3
    return g

df = df.groupby(FIXED_BASELINE_KEYS, group_keys=False).apply(_fixed_baseline)

x = df["이용량"].astype(float).values
fixed_num = np.select(
    [x <= df["_fq1"].values,
     (x > df["_fq1"].values) & (x <= df["_fq2"].values),
     (x > df["_fq2"].values) & (x <= df["_fq3"].values),
     x > df["_fq3"].values],
    [1, 2, 3, 4],
    default=1
).astype(int)
df["fixed_level_num"] = fixed_num
df["fixed_level"] = df["fixed_level_num"].map(LEVEL_LABEL)

# 5) 최종 혼잡도/등급
df["혼잡도"] = (df["min_max"] * 100).round(2)                          # 0~100
final_num = np.rint(W_Q*df["quantile_level_num"] + W_F*df["fixed_level_num"]).astype(int)
df["final_level_num"] = final_num.clip(1, 4)
df["final_level"] = df["final_level_num"].map(LEVEL_LABEL)

# 6) 저장(엑셀만) — 혼잡도 최종등급_통합.xlsx
#  기존 "혼잡도 최종등급_통합.csv"와 유사한 컬럼 구성(년도는 제외해 호환 유지)
cols_out = ["시도","시군구","읍면동","시간대","혼잡도","z_score","min_max","fixed_level","quantile_level","final_level"]

# 시간대 정렬
time_order = {f"{h:02d}시": h for h in range(24)}
out = df.copy()

# 시간대 정규화 → 카테고리 순서 지정
hours = [f"{h:02d}시" for h in range(24)]
out["시간대"] = (out["시간대"].astype(str)
                 .str.extract(r"(\d{1,2})")[0].astype(int)
                 .map(lambda h: f"{h:02d}시"))
out["시간대"] = pd.Categorical(out["시간대"], categories=hours, ordered=True)

# 년도 숫자화
out["년도"] = pd.to_numeric(out["년도"], errors="coerce")

# key 없이 정상 정렬
out.sort_values(["시도","시군구","읍면동","년도","시간대"], inplace=True)

out = out[cols_out].copy()

with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as w:
    out.to_excel(w, sheet_name="통합", index=False)

print(f"[DONE] 저장: {OUT_XLSX} (rows={len(out)})")


C:\Users\hyunj\AppData\Local\Temp\ipykernel_29916\4260706580.py:73: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(GROUP_KEYS, group_keys=False).apply(_group_metrics)
C:\Users\hyunj\AppData\Local\Temp\ipykernel_29916\4260706580.py:85: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(FIXED_BASELINE_KEYS, group_keys=False).apply(_fixed_baseline)


[DONE] 저장: 혼잡도 최종등급_통합.xlsx (rows=93384)


In [1]:
import pandas as pd

file_path = "혼잡도 최종등급_통합.xlsx"
OUT_XLSX = "혼잡도 최종_법정동_등급_통합.xlsx"
out = pd.read_excel(file_path, sheet_name="통합")